## Demand CAGR for Home Services by State

In [1]:
import pandas as pd
import numpy as np

print("🔄 Loading data...")
df = pd.read_csv("hs_states_demand_2021-2025.csv")
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

# Fixed service list
services = [
    "Asphalt/concrete work",
    "Driveway sealing",
    "Gutter cleaning",
    "Junk removal",
    "Pressure washing",
    "Soft washing",
    "Tree service",
    "Window cleaning",
]

print(f"✅ Data loaded: {df.shape[0]:,} rows, {len(services)} services")
print(f"📅 Years: {sorted(df['year'].unique())}")
print(f"🗺️ States: {len(df['geo'].unique())}")

# Step 1: Annual averages per state and year
annual_avg = (
    df.groupby(["geo", "year"])[services]
    .mean()
    .reset_index()
)

print(f"✅ Annual averages computed: {annual_avg.shape[0]} rows")

# Step 2: Get 2021 (start) and 2025 (end) states x services
start_data = annual_avg[annual_avg["year"] == 2021].set_index("geo")[services]
end_data = annual_avg[annual_avg["year"] == 2025].set_index("geo")[services]

# Step 3: CAGR calculation function
def calc_cagr_matrix(end, start, years=4):
    """
    Compute CAGR matrix over `years` years:
    CAGR = (end / start)^(1/years) - 1

    Args:
        end: (n_states, n_services) DataFrame, 2025 values
        start: (n_states, n_services) DataFrame, 2021 values
        years: number of years between start and end (2021–2025 → 4 years)

    Returns:
        DataFrame of CAGR values (same shape as `end`), in decimal (0–1 form).
    """
    # Start matrix (n_states x n_services)
    start_vals = start.to_numpy()
    end_vals   = end.to_numpy()

    # Ensure positive values
    start_vals = np.where(start_vals <= 0, 1.0, start_vals)
    end_vals   = np.where(end_vals <= 0, 1.0, end_vals)

    # CAGR in decimal
    cagr = (end_vals / start_vals) ** (1.0 / years) - 1.0

    return pd.DataFrame(
        cagr,
        index=end.index,
        columns=end.columns
    )

# Step 4: Compute CAGR (2021–2025, 4 years)
cagr_decimal = calc_cagr_matrix(end_data, start_data, years=4)

# Convert to percentage (0–100 scale)
cagr_pct = cagr_decimal * 100.0

print("✅ CAGR (2021–2025) calculated!")
print(f"📈 CAGR range (pct): {cagr_pct.min().min():.2f}% to {cagr_pct.max().max():.2f}%")


# Step 5: Analyze one service (e.g., Junk removal)
print("\n" + "="*80)
print("🏆 TOP 10 STATES BY JUNK REMOVAL CAGR (2021–2025)")
print("="*80)
print(cagr_pct["Junk removal"].sort_values(ascending=False).head(10).round(2))

# Step 6: Summary stats
print("\n📊 CAGR SUMMARY STATS (pct points):")
print(cagr_pct.describe().round(3))


# Step 7: Save CAGR table for ZIP‑level analyzer
df_cagr_detailed = cagr_pct.reset_index()
df_cagr_detailed = df_cagr_detailed.round(3)
df_cagr_detailed.to_csv("demand_cagr_by_state.csv", index=False)

print("\n" + "="*80)
print("💾 SAVED: demand_cagr_by_state.csv")
print(f"📋 Columns: {list(df_cagr_detailed.columns)}")
print("✅ CAGR table READY FOR ZIP CROSSWALK MERGE!")

print("\nSample (CAGR %):")
print(df_cagr_detailed[["geo", "Junk removal", "Tree service", "Window cleaning"]].head(10))


# Step 8: Hot states (high average CAGR across services)
print("\n🔥 HOT STATES (avg CAGR > 3.0%):")
avg_cagr = cagr_pct.mean(axis=1).round(3)
hot_states = avg_cagr[avg_cagr > 3.0].sort_values(ascending=False)
print(hot_states.head(10))


🔄 Loading data...
✅ Data loaded: 13,100 rows, 8 services
📅 Years: [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
🗺️ States: 50
✅ Annual averages computed: 300 rows
✅ CAGR (2021–2025) calculated!
📈 CAGR range (pct): -47.56% to 254.54%

🏆 TOP 10 STATES BY JUNK REMOVAL CAGR (2021–2025)
geo
HI    142.24
ID    100.84
UT     62.56
LA     60.98
ME     57.50
NM     52.28
OK     51.67
NH     46.97
AR     43.20
NE     38.06
Name: Junk removal, dtype: float64

📊 CAGR SUMMARY STATS (pct points):
       Asphalt/concrete work  Driveway sealing  Gutter cleaning  Junk removal  \
count                 50.000            50.000           50.000        50.000   
mean                   0.733            -5.941           14.213        19.988   
std                   12.056            18.770           19.790        28.770   
min                  -27.088           -47.563          -27.954       -27.271   
25%                    0.000           -15.082          